In [51]:
import os
#import pandas as pd
#import re
#import numpy as np
from datetime import datetime
import warnings
import pathlib 
from datetime import datetime

warnings.filterwarnings("ignore", category=UserWarning, module='openpyxl')

# MANUAL input to change
your_ing_id = "PY40DL"
date = datetime.strptime('2024', "%Y")
edition = "ESG v2"
combine_intermediate_files = False

# DO NOT CHANGE ANYTHING BELOW - only when developing code ##########################################################################
pap_categories = ["New financial product/-service/channel (PAP)", "Termination of financial product/-service/channel (PAP)", "Significant change financial product/-service/channel (PAP)"]
non_pap_categories = ["Risk assessment (non-PAP)", "Other change (non-PAP)"]


# determining operating system
is_windows = os.name == 'nt'
if is_windows:
    base_dir = os.path.join("C:", "\\Users", your_ing_id, "ING")
else:
    base_dir = pathlib.Path(f"/Users/{your_ing_id}/Library/CloudStorage/OneDrive-SharedLibraries-ING")

# Define paths dynamically
directory_path0 = os.getcwd()
directory_path = os.path.join(base_dir, "Product Evaluation and Risk Assessment Library (PEARL) - PEARL_Repository")
directory_path2 = os.path.join(base_dir, "Product Evaluation and Risk Assessment Library (PEARL) - MI Dashboard")
directory_path3 = os.path.join(directory_path0, "Intermediate results", edition)
directory_path4 = os.path.join(directory_path0, "Intermediate results combined", edition)
directory_path5 = os.path.join(directory_path0, "Final overview")
dir_output = os.path.join(os.getcwd(), "Output", edition)


if not os.path.exists(directory_path4):
    os.makedirs(directory_path4)

if not os.path.exists(dir_output):
    os.makedirs(dir_output)


final_overview = pd.read_csv(os.path.join(directory_path5, 'final_overview ' + edition +'.csv'), sep=';')
final_overview["Start Date"] = pd.to_datetime(final_overview["Start Date"])  # Convert to datetime
final_overview["End Date"] = pd.to_datetime(final_overview["End Date"])  # Convert to datetime

if is_windows:
    final_overview['Folder'] = final_overview['Folder'].str.replace('/', '\\')


# Print paths
print(f"Operating System: {'Windows' if is_windows else 'Mac/Linux'}")
print(f"PEARL Repository Path: {directory_path}")
print(f"MI Dashboard Path: {directory_path2}")
print(f"Intermediate Results Path: {directory_path3}")

Operating System: Windows
PEARL Repository Path: C:\Users\PY40DL\ING\Product Evaluation and Risk Assessment Library (PEARL) - PEARL_Repository
MI Dashboard Path: C:\Users\PY40DL\ING\Product Evaluation and Risk Assessment Library (PEARL) - MI Dashboard
Intermediate Results Path: c:\Users\PY40DL\OneDrive - ING\Projects\Risk Assesments\Intermediate results\ESG v2


In [52]:
def drop_duplicate_RA(df):

    df['File date'] = pd.to_datetime(df['File date'])
    df = df.sort_values(by='File date', ascending=False)

    latest_dates = df.groupby('File')['File date'].max().reset_index()

    df = df.merge(latest_dates, on=['File', 'File date'])

    df.reset_index(drop=True, inplace=True)
    return(df)

In [53]:

if combine_intermediate_files:
    # Initialize DataFrames
    dt1, dt2, dt3, dt4 = [pd.DataFrame() for _ in range(4)]

    # Check if directory is empty
    all_files = os.listdir(directory_path3)
    if not all_files:
        raise ValueError("Error: directory_path3 is empty!")

    # Process files
    for f in tqdm(all_files):
        file_path = os.path.join(directory_path3, f)

        # Ensure it's a file, not a directory
        if os.path.isfile(file_path):
            try:
                dt_f = pd.read_csv(file_path, sep=';', encoding='utf-8')
                if 'journey_summary' in f:
                    dt1 = pd.concat([dt1, dt_f])
                elif 'module_selected' in f:
                    dt2 = pd.concat([dt2, dt_f])
                elif 'ESG_answers' in f:
                    dt3 = pd.concat([dt3, dt_f])
                elif 'ORM_ESG_challange' in f:
                    dt4 = pd.concat([dt4, dt_f])

            except Exception as e:
                print(f"Error processing file {f}: {e}")


    #drop duplicates
    dt1 = drop_duplicate_RA(dt1)
    dt2 = drop_duplicate_RA(dt2)
    dt3 = drop_duplicate_RA(dt3)
    dt4 = drop_duplicate_RA(dt4)
    dt2 = dt2.rename(columns={'End Date': 'Module End Date'})


# Normalize folder paths (Mac-safe)
#for i in range(len(dataframes)):
 #   if 'Folder' in dataframes[i].columns:
  #      dataframes[i]['Folder'] = dataframes[i]['Folder'].apply(lambda x: str(Path(x)))


    # merge with final overview each 
    #dt1 = dt1.merge(final_overview, on="Folder", how="right")
    dt2 = dt2.merge(final_overview, on="Folder", how="right")
    #dt3 = dt3.merge(final_overview, on="Folder", how="right")
    #dt4 = dt4.merge(final_overview, on="Folder", how="right")

    dt1.to_csv(os.path.join(directory_path4, 'dt1.csv'), sep = ';')
    dt2.to_csv(os.path.join(directory_path4, 'dt2.csv'), sep = ';')
    dt3.to_csv(os.path.join(directory_path4, 'dt3.csv'), sep = ';')
    dt4.to_csv(os.path.join(directory_path4, 'dt4.csv'), sep = ';')

else:
    dt1 = pd.read_csv(os.path.join(directory_path4, 'dt1.csv'), sep = ';')
    dt2 = pd.read_csv(os.path.join(directory_path4, 'dt2.csv'), sep = ';')
    dt3 = pd.read_csv(os.path.join(directory_path4, 'dt3.csv'), sep = ';')
    dt4 = pd.read_csv(os.path.join(directory_path4, 'dt4.csv'), sep = ';')

C:\Users\PY40DL\AppData\Local\Temp\ipykernel_17784\3836036350.py:58: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  dt2 = pd.read_csv(os.path.join(directory_path4, 'dt2.csv'), sep = ';')


In [54]:
# ESG module selection dummy

module_selection = dt3[['File', 'Folder']].drop_duplicates()
module_selection['ESG Module exists 2'] = True

dt2 = dt2.merge(module_selection, how = 'left', on = ['File', 'Folder'])

cond1 = dt2['Module Title'] == 'ESG Module'
cond11 = dt2['ESG Module exists 2'] == True
cond2 = dt2['Applicable'] == True

dt2['ESG Module exists'] = cond1 | cond11
dt2['ESG module selected'] = cond1 & cond2

cond3 = dt2['File'].isna()
dt2['File Missing'] = cond3


dt2_grouped = dt2.groupby(
    ["Folder", "File", "ID", "Process Category", "Tribe", "Status",'Start Date', "End Date", "File Missing"],
    dropna=False
).agg({
    'ESG Module exists': 'max',
    'ESG module selected': 'max'
}).reset_index()


In [55]:
cond = dt1['Parameter'].isin(['Product', 'Product Category'])

dt1_filtered = dt1[cond][['Parameter', 'File', 'Folder', 'Value']].drop_duplicates()
#dt1_filtered['dummy'] = 1


dt1_filtered = dt1_filtered.pivot(index=['Folder', 'File'], columns='Parameter', values='Value')
dt1_filtered.reset_index(inplace=True)


In [56]:
final_output = dt2_grouped.merge(dt1_filtered, how = 'left', on = ['File', 'Folder'])


dt4_selected = dt4[['Name', 'Sanity check', 'Overall opinion', 'File', 'Folder']]

final_output = final_output.merge(dt4_selected, how = 'left', on = ['File', 'Folder'])

cond4 = final_output['Overall opinion'].isna() == False

final_output.loc[cond4, 'ESG Module exists'] = True
final_output.loc[cond4, 'ESG module selected'] = True

final_output.to_csv('ESG_output.csv', sep = ';', index=None)

In [57]:
cond5 = (dt3['Answer'].isna() == False) | (dt3['Description'].isna() == False)

dt3 = dt3[cond5]

keys = final_output[['ID', 'File', 'Folder', 'Product', 'Product Category', 'Tribe', 'Status']]

dt3 = dt3.merge(keys, how = 'left', on = ['File', 'Folder'])

dt3.to_csv('ESG_output2.csv', sep = ';', index=None)